In [3]:
# ============================================================================
# IPA via MAX SECOND DERIVATIVE — standalone, single cell
#
# Pipeline (raw data only, no curve fitting):
#   1. Auto-detect pruning levels from p-percentage_* directories in BASE_DIR
#   2. For each (P%, BS): load averaged_runs_p_{p}_bs_{bs}.csv
#   3. Step-artifact truncation: cut at first BN >= BN_STEP_MIN where
#      |CE[i] - CE[i-1]| > STEP_THRESH (early-stopping averaging artifact)
#   4. Elbow = data point with max discrete second derivative
#      (central differences, np.gradient twice; endpoints excluded).
#      SMOOTH_W = 1 -> raw data; odd int > 1 -> moving average first
#      (edge points additionally excluded from the argmax search)
#   5. IPA = |CE_o - CE_learned| / BN_learned,  CE_o = ln(10)
#      Degenerate flat curves (e.g. P=100%) -> NaN
#   6. Write summary CSV + IPA-vs-P% plot
#
# Requires: numpy, pandas, matplotlib
# ============================================================================
import os, glob, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Config ────────────────────────────────────────────────────────────────────
BN_STEP_MIN = 100     # step detection starts here (shared convention)
STEP_THRESH = 0.01    # CE jump that flags the averaging artifact
SMOOTH_W    = 30       # 1 = raw data; odd int > 1 = moving-average smoothing
CE_o        = np.log(10)
BATCH_SIZES = [64, 1024, 60000]
BS_COLOR    = {64: "#1f77b4", 1024: "#d62728", 60000: "#2ca02c"}

BASE_DIR = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-FMNIST\prune_layers_ALL"
OUT_DIR  = r"C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-FMNIST\result"
# ──────────────────────────────────────────────────────────────────────────────

PRUNING_LEVELS = sorted(float(re.search(r"p-percentage_([\d.]+)", d).group(1))
                        for d in glob.glob(os.path.join(BASE_DIR, "p-percentage_*")))
print(f"Found {len(PRUNING_LEVELS)} pruning levels  |  SMOOTH_W={SMOOTH_W} "
      f"({'raw data' if SMOOTH_W <= 1 else 'moving average'})")


def load_curve(p, bs):
    """Averaged CE curve for (p, bs), truncated at the step artifact."""
    f = os.path.join(BASE_DIR, f"p-percentage_{p}", f"batch_size_{bs}",
                     f"averaged_runs_p_{p}_bs_{bs}.csv")
    if not os.path.exists(f):
        return None
    df = pd.read_csv(f)
    df.columns = df.columns.str.strip()
    ce_col = next((c for c in df.columns if c in ("Avg_CE_Test", "Avg_CE_test")), None)
    bn_col = next((c for c in df.columns if "Batch" in c), None)
    if ce_col is None or bn_col is None:
        return None
    df = df.dropna(subset=[ce_col, bn_col])
    bns = df[bn_col].values.astype(float)
    ces = df[ce_col].values.astype(float)
    cutoff_BN = float(bns[-1])
    for i in range(1, len(bns)):
        if bns[i] >= BN_STEP_MIN and abs(ces[i] - ces[i - 1]) > STEP_THRESH:
            cutoff_BN = float(bns[i])
            break
    m = bns < cutoff_BN
    return bns[m], ces[m]


def second_derivative_elbow(BN, CE, smooth_w=SMOOTH_W):
    """Max-second-derivative elbow on the data. -> (BN_learned, CE_learned, IPA)"""
    BN = np.asarray(BN, float)
    CE = np.asarray(CE, float)
    if len(BN) < max(5, smooth_w + 2) or np.ptp(CE) <= 1e-10:
        return np.nan, np.nan, np.nan                 # degenerate (e.g. P=100%)
    if smooth_w > 1:
        y = np.convolve(CE, np.ones(int(smooth_w)) / smooth_w, mode="same")
        h = int(smooth_w) // 2                        # unreliable convolution edges
    else:
        y, h = CE, 0
    d2 = np.gradient(np.gradient(y, BN), BN)          # discrete 2nd derivative
    lo, hi = 1 + h, len(BN) - 1 - h                   # exclude endpoints (+ edges)
    if hi <= lo:
        return np.nan, np.nan, np.nan
    i = lo + int(np.argmax(d2[lo:hi]))
    if BN[i] <= 0:
        return np.nan, np.nan, np.nan
    return float(BN[i]), float(CE[i]), abs(CE_o - CE[i]) / BN[i]


# ── Sweep all (P%, BS), build summary ─────────────────────────────────────────
rows = []
for p in PRUNING_LEVELS:
    row = {"P%": p * 100}
    for bs in BATCH_SIZES:
        curve = load_curve(p, bs)
        bn_l, ce_l, ipa = second_derivative_elbow(*curve) if curve is not None else (np.nan,)*3
        row[f"BN_learned_{bs}"] = bn_l
        row[f"CE_learned_{bs}"] = ce_l
        row[f"IPA_Avg_{bs}"]    = ipa
    rows.append(row)
summary_df = pd.DataFrame(rows)

csv_path = os.path.join(OUT_DIR, "ipa_summary_2nd_deriv_standalone.csv")
summary_df.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")
print(summary_df[["P%"] + [c for b in BATCH_SIZES for c in (f"BN_learned_{b}", f"IPA_Avg_{b}")]]
      .to_string(index=False))

# ── Plot IPA vs P% ────────────────────────────────────────────────────────────
plt.rcParams.update({"font.size": 13})
fig, ax = plt.subplots(figsize=(9, 5.5))
for bs in BATCH_SIZES:
    sub = summary_df.dropna(subset=[f"IPA_Avg_{bs}"])
    ax.plot(sub["P%"], sub[f"IPA_Avg_{bs}"], "^-.", color=BS_COLOR[bs],
            ms=6, lw=2, label=f"BS={bs}")
ax.set_xlabel("Pruning Percentage (%)")
ax.set_ylabel("IPA")
smooth_tag = "raw data" if SMOOTH_W <= 1 else f"moving avg w={SMOOTH_W}"
ax.set_title(f"IPA vs Pruning — max 2nd derivative elbow ({smooth_tag})")
ax.grid(True, alpha=0.3)
ax.legend(frameon=False)
png_path = os.path.join(OUT_DIR, "ipa_plot_2nd_deriv_standalone.png")
plt.tight_layout()
plt.savefig(png_path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {png_path}")


Found 11 pruning levels  |  SMOOTH_W=30 (moving average)
Saved: C:\Users\Student\Desktop\Projects\research\physlab\SLP\SLP-FMNIST\result\ipa_summary_2nd_deriv_standalone.csv
   P%  BN_learned_64  IPA_Avg_64  BN_learned_1024  IPA_Avg_1024  BN_learned_60000  IPA_Avg_60000
  0.0           17.0    0.081979             17.0      0.085980              17.0       0.085278
 10.0           17.0    0.082038             17.0      0.086239              17.0       0.085831
 20.0           17.0    0.082245             17.0      0.086615              17.0       0.085968
 30.0           17.0    0.081734             17.0      0.086571              17.0       0.085925
 40.0           17.0    0.081435             17.0      0.086183              17.0       0.085515
 50.0           17.0    0.079707             17.0      0.085281              17.0       0.085128
 60.0           17.0    0.077091             17.0      0.083460              17.0       0.083354
 70.0           17.0    0.072468             17.0 